# Modul 1 · Pemanfaatan Algoritma AI untuk Pengolahan Data
**Pelatihan Machine Learning & Data Analysis**

Sebelum AI bisa "belajar", data harus diolah dulu. Prinsip penting:

> **"Garbage In, Garbage Out"** — data kotor menghasilkan analisis yang salah.

Di modul ini kita berlatih alur pengolahan data yang paling sering dipakai:

1. **Membaca & mengenali data** (load + inspeksi)
2. **Menemukan masalah pada data** (kosong, duplikat, tidak konsisten)
3. **Membersihkan data** (cleaning)
4. **Merangkum data** (agregasi → dari deskriptif ke insight)

> ⚠️ Prasyarat: notebook `00_generate_dataset.ipynb` sudah dijalankan.

In [ ]:
import pandas as pd
from pathlib import Path

DIR = Path.cwd().parent
pd.set_option("display.width", 120)

## LANGKAH 1 : MEMBACA & MENGENALI DATA

In [ ]:
df = pd.read_csv(DIR / "dataset" / "data_transaksi_sparepart.csv")

print(f"\nJumlah baris  : {len(df)}")
print(f"Jumlah kolom  : {len(df.columns)}")
print("\n5 baris pertama:")
print(df.head())

print("\nInfo tipe data tiap kolom:")
print(df.dtypes)

## LANGKAH 2 : MENEMUKAN MASALAH PADA DATA

In [ ]:
print("\nJumlah data KOSONG per kolom:")
print(df.isna().sum())

print(f"\nJumlah baris DUPLIKAT : {df.duplicated().sum()}")

print("\nPenulisan nama part yang TIDAK KONSISTEN (contoh):")
print(sorted(df["nama_part"].unique()))

## LANGKAH 3 : MEMBERSIHKAN DATA

In [ ]:
jumlah_awal = len(df)

# 3a. Hapus baris duplikat
df = df.drop_duplicates()
print(f"\n3a. Hapus duplikat        : {jumlah_awal} -> {len(df)} baris")

# 3b. Rapikan penulisan teks: "OLI_MESIN" -> "Oli Mesin"
df["nama_part"] = (df["nama_part"]
                   .str.replace("_", " ")
                   .str.title()
                   .str.strip())
print(f"3b. Nama part konsisten   : {sorted(df['nama_part'].unique())}")

# 3c. Tangani data kosong
#     - dealer kosong  -> isi dengan label khusus (tetap bisa dianalisis)
#     - qty kosong     -> baris dihapus (nilai inti transaksi tidak boleh ditebak)
df["dealer"] = df["dealer"].fillna("Tidak Tercatat")
sebelum = len(df)
df = df.dropna(subset=["qty"])
print(f"3c. Data kosong           : dealer diisi label, "
      f"{sebelum - len(df)} baris tanpa qty dihapus")

# 3d. Perbaiki tipe data & hitung ulang kolom total agar konsisten
df["qty"] = df["qty"].astype(int)
df["tanggal"] = pd.to_datetime(df["tanggal"])
df["total"] = df["qty"] * df["harga_satuan"]
print(f"3d. Tipe data dirapikan   : qty -> int, tanggal -> datetime")

print(f"\nHasil akhir: {len(df)} baris data BERSIH siap dianalisis.")

## LANGKAH 4 : MERANGKUM DATA (DESKRIPTIF -> INSIGHT)

In [ ]:
print("\nTotal penjualan per jenis part (Rp):")
ringkasan_part = (df.groupby("nama_part")["total"]
                    .agg(jumlah_transaksi="count", total_rupiah="sum")
                    .sort_values("total_rupiah", ascending=False))
print(ringkasan_part.to_string(float_format=lambda x: f"{x:,.0f}"))

print("\nTotal penjualan per dealer (Rp):")
ringkasan_dealer = (df.groupby("dealer")["total"].sum()
                      .sort_values(ascending=False))
print(ringkasan_dealer.to_string(float_format=lambda x: f"{x:,.0f}"))

print("\nTren penjualan per bulan (Rp):")
per_bulan = df.groupby(df["tanggal"].dt.to_period("M"))["total"].sum()
print(per_bulan.to_string(float_format=lambda x: f"{x:,.0f}"))

# Simpan data bersih untuk dipakai modul berikutnya (deteksi anomali)
file_bersih = DIR / "dataset" / "data_transaksi_bersih.csv"
df.to_csv(file_bersih, index=False)
print(f"\nData bersih disimpan ke: {file_bersih.name}")

print("""
--------------------------------------------------------------------------------
KESIMPULAN MODUL 1
--------------------------------------------------------------------------------
1. Data mentah hampir selalu kotor: ada yang kosong, duplikat, tidak konsisten.
2. Pembersihan data = fondasi. Tanpa ini, model AI akan belajar dari kesalahan.
3. Agregasi (groupby) mengubah ribuan baris menjadi ringkasan yang bisa
   langsung dipakai mengambil keputusan -> ini level analitik DESKRIPTIF.
4. Modul berikutnya: naik level ke PREDIKTIF dengan Machine Learning.
--------------------------------------------------------------------------------
""")

---
### 💡 Latihan mandiri
1. Tambahkan agregasi baru: total penjualan per **dealer per bulan** — dealer mana yang trennya turun?
2. Ubah strategi penanganan data kosong: alih-alih membuang baris `qty` kosong, coba isi dengan **median qty per jenis part**. Bandingkan hasil rangkumannya.

In [ ]:
import numpy as np
# Agregasi baru: total penjualan per dealer per bulan
dealer_month = (df.groupby(['dealer', df['tanggal'].dt.to_period('M')])['total']
                 .sum()
                 .unstack(fill_value=0))
# Pastikan kolom terurut secara kronologis
dealer_month = dealer_month.reindex(sorted(dealer_month.columns), axis=1)
print("Total penjualan per dealer per bulan (sample):")
print(dealer_month.head(10).to_string(float_format=lambda x: f"{x:,.0f}"))

# Hitung tren sederhana: slope dari regresi linier terhadap indeks waktu
slopes = {}
for dealer, row in dealer_month.iterrows():
    y = row.values.astype(float)
    if y.sum() == 0 or len(y) < 2:
        slopes[dealer] = 0.0
    else:
        x = np.arange(len(y))
        slopes[dealer] = np.polyfit(x, y, 1)[0]
slopes = pd.Series(slopes).sort_values()

print("\nDealer dengan tren menurun (slope < 0):")
print(slopes[slopes < 0].to_string(float_format=lambda x: f"{x:,.2f}"))

# Perkiraan perubahan relatif antara bulan -3 dan bulan terakhir untuk tambahan konteks
if dealer_month.shape[1] >= 3:
    last3 = dealer_month.iloc[:, -3:]
    pct = (last3.iloc[:, -1] - last3.iloc[:, 0]) / last3.iloc[:, 0].replace({0: np.nan})
    pct = pct.fillna(0).sort_values()
    print("\nPerkiraan % perubahan dari bulan -3 ke bulan terakhir (per dealer):")
    print(pct.to_string(float_format=lambda x: f"{x:.2%}"))
else:
    print("Tidak cukup bulan untuk menghitung perubahan 3-bulan.")


In [ ]:
# ---- STRATEGI ALTERNATIF: isi qty kosong dengan median per jenis part ----
# Bandingkan hasil strategi asli (hapus) vs baru (isi median) secara independen
import pandas as pd
from pathlib import Path
DIR = Path.cwd().parent
df_mentah = pd.read_csv(DIR / "dataset" / "data_transaksi_sparepart.csv")

# step umum: drop duplikat & rapiin nama_part
df_mentah = df_mentah.drop_duplicates()
df_mentah["nama_part"] = df_mentah["nama_part"].str.replace("_", " ").str.title().str.strip()

# STRATEGI 1: hapus baris qty kosong
df_hapus = df_mentah.copy()
df_hapus["dealer"] = df_hapus["dealer"].fillna("Tidak Tercatat")
df_hapus = df_hapus.dropna(subset=["qty"])
df_hapus["qty"] = df_hapus["qty"].astype(int)
df_hapus["tanggal"] = pd.to_datetime(df_hapus["tanggal"])
df_hapus["total"] = df_hapus["qty"] * df_hapus["harga_satuan"]

print("=== STRATEGI ASLI: hapus baris qty kosong ===")
print(f"Baris: {len(df_hapus)} | Rata-rata total: {df_hapus["total"].mean():,.0f}")
ringkasan_part_hapus = df_hapus.groupby("nama_part")["total"].agg(jumlah_transaksi="count", total_rupiah="sum")
ringkasan_dealer_hapus = df_hapus.groupby("dealer")["total"].sum()

# STRATEGI 2: isi qty kosong dengan median per jenis part
df_isi = df_mentah.copy()
df_isi["dealer"] = df_isi["dealer"].fillna("Tidak Tercatat")
median_qty = df_isi[df_isi["qty"].notna()].groupby("nama_part")["qty"].median().round().astype(int)
mask = df_isi["qty"].isna()
df_isi.loc[mask, "qty"] = df_isi.loc[mask, "nama_part"].map(median_qty)
df_isi["qty"] = df_isi["qty"].astype(int)
df_isi["tanggal"] = pd.to_datetime(df_isi["tanggal"])
df_isi["total"] = df_isi["qty"] * df_isi["harga_satuan"]

print("\n=== STRATEGI BARU: isi qty kosong dengan median per jenis part ===")
print(f"Baris: {len(df_isi)} | Rata-rata total: {df_isi["total"].mean():,.0f}")
print("Median qty per part:")
print(median_qty.to_string())
ringkasan_part_isi = df_isi.groupby("nama_part")["total"].agg(jumlah_transaksi="count", total_rupiah="sum")
ringkasan_dealer_isi = df_isi.groupby("dealer")["total"].sum()

print("\nPERBANDINGAN TOTAL PENJUALAN PER PART:")
banding = pd.DataFrame({"asli_hapus": ringkasan_part_hapus["total_rupiah"], "baru_isi_median": ringkasan_part_isi["total_rupiah"]})
banding["selisih"] = banding["baru_isi_median"] - banding["asli_hapus"]
banding["%_ubah"] = ((banding["baru_isi_median"] - banding["asli_hapus"]) / banding["asli_hapus"] * 100).round(2)
print(banding.to_string(float_format=lambda x: f"{x:,.0f}" if isinstance(x, (int, float)) else str(x)))

print("\nPERBANDINGAN TOTAL PENJUALAN PER DEALER:")
banding_d = pd.DataFrame({"asli_hapus": ringkasan_dealer_hapus, "baru_isi_median": ringkasan_dealer_isi})
banding_d["selisih"] = banding_d["baru_isi_median"] - banding_d["asli_hapus"]
banding_d["%_ubah"] = ((banding_d["baru_isi_median"] - banding_d["asli_hapus"]) / banding_d["asli_hapus"] * 100).round(2)
print(banding_d.to_string(float_format=lambda x: f"{x:,.0f}" if isinstance(x, (int, float)) else str(x)))
